# Part 2 — SQL Basics to Advanced

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 32: What is a Database? — Tables, Keys, and the Read Replica](#chapter_32_what_is_a_database_tables_keys_and_the_read_replica)
- [Chapter 33: SELECT and WHERE — Reading Data from a Database](#chapter_33_select_and_where_reading_data_from_a_database)
- [Chapter 34: GROUP BY and HAVING — Aggregation in SQL](#chapter_34_group_by_and_having_aggregation_in_sql)
- [Chapter 35: SQL JOINs — Combining Tables](#chapter_35_sql_joins_combining_tables)
- [Chapter 36: Subqueries and CTEs — Queries Inside Queries](#chapter_36_subqueries_and_ctes_queries_inside_queries)
- [Chapter 37: Window Functions — Aggregations Without Collapsing Rows](#chapter_37_window_functions_aggregations_without_collapsing_rows)
- [Chapter 38: Writing Queries for Real Datasets — Putting It All Together](#chapter_38_writing_queries_for_real_datasets_putting_it_all_together)
- [Chapter 39: SQL Data Types, Casting, and Date Functions](#chapter_39_sql_data_types_casting_and_date_functions)
- [Chapter 40: Indexes and Query Performance](#chapter_40_indexes_and_query_performance)
- [Chapter 41: DDL, DML, and Views — The Write Side of SQL](#chapter_41_ddl_dml_and_views_the_write_side_of_sql)
- [Chapter 41a: NoSQL Databases — When the Table Doesn't Fit](#chapter_41a_nosql_databases_when_the_table_doesn_t_fit)

---

# Chapter 32: What is a Database? — Tables, Keys, and the Read Replica

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### Tables, Rows, and Columns

```sql
-- CinemaStream users table schema
CREATE TABLE users (
    user_id        INTEGER     PRIMARY KEY,
    name           TEXT        NOT NULL,
    email          TEXT        UNIQUE NOT NULL,
    country        TEXT        NOT NULL,
    language_pref  TEXT,
    plan           TEXT        NOT NULL DEFAULT 'Free',
    signup_date    DATE        NOT NULL,
    churned        BOOLEAN     NOT NULL DEFAULT FALSE
);
```

### Primary Keys and Foreign Keys

```sql
-- watch_events references users and movies
CREATE TABLE watch_events (
    event_id       INTEGER     PRIMARY KEY,
    user_id        INTEGER     NOT NULL REFERENCES users(user_id),
    movie_id       INTEGER     NOT NULL REFERENCES movies(movie_id),
    watch_started  TIMESTAMP   NOT NULL,
    watch_minutes  INTEGER,
    completed      BOOLEAN,
    device         TEXT,
    country        TEXT
);
```

### Connecting to SQLite in Python

In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Create and populate tables from the CinemaStream CSVs
users_df  = pd.read_csv("cinemastream/data/users.csv",        encoding="utf-8")
movies_df = pd.read_csv("cinemastream/data/movies.csv",       encoding="utf-8")
events_df = pd.read_csv("cinemastream/data/watch_events.csv", encoding="utf-8")

users_df.to_sql("users",        conn, index=False, if_exists="replace")
movies_df.to_sql("movies",      conn, index=False, if_exists="replace")
events_df.to_sql("watch_events", conn, index=False, if_exists="replace")

print("Tables loaded into SQLite")

### Running SQL Queries

In [ ]:
# Execute a query, return as DataFrame
def query(sql: str, conn) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

# Show all tables
result = query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(result)

In [ ]:
# Check row counts
for table in ["users", "movies", "watch_events"]:
    count = query(f"SELECT COUNT(*) as n FROM {table}", conn)
    print(f"{table}: {count['n'][0]} rows")

### SQL vs NoSQL — Key Differences

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Set up the demo database (in production this would be a PostgreSQL read replica)
conn = sqlite3.connect(":memory:")

for table_name, csv_file in [
    ("users",        "users.csv"),
    ("movies",       "movies.csv"),
    ("watch_events", "watch_events.csv"),
]:
    df = pd.read_csv(f"cinemastream/data/{csv_file}", encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def query(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

# Explore the schema
schema_q = """
SELECT 
    name AS table_name,
    sql  AS create_statement
FROM sqlite_master 
WHERE type='table'
ORDER BY name
"""
tables = query(schema_q)
for _, row in tables.iterrows():
    print(f"\n--- {row['table_name']} ---")
    print(row['create_statement'])

In [ ]:
# Quick business verification: do our tables make sense?
print("=== CinemaStream Database Health Check ===")

# Users
users_check = query("""
SELECT 
    COUNT(*) as total_users,
    SUM(CASE WHEN churned = 1 THEN 1 ELSE 0 END) as churned_count,
    COUNT(DISTINCT plan) as distinct_plans,
    COUNT(DISTINCT country) as distinct_countries
FROM users
""")
print(f"\nUsers: {users_check.to_dict('records')[0]}")

# Watch events
events_check = query("""
SELECT
    COUNT(*) as total_events,
    COUNT(DISTINCT user_id) as unique_users_watching,
    MIN(watch_minutes) as min_minutes,
    MAX(watch_minutes) as max_minutes,
    ROUND(AVG(watch_minutes), 1) as avg_minutes
FROM watch_events
""")
print(f"Events: {events_check.to_dict('records')[0]}")

# Movies
movies_check = query("""
SELECT 
    COUNT(*) as total_movies,
    COUNT(DISTINCT genre) as genres,
    COUNT(DISTINCT original_lang) as languages
FROM movies
""")
print(f"Movies: {movies_check.to_dict('records')[0]}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

```sql
CREATE TABLE subscriptions (
    subscription_id INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL REFERENCES users(user_id),
    plan            TEXT    NOT NULL CHECK (plan IN ('Free','Basic','Premium')),
    end_date        DATE,
    amount_sgd      DECIMAL NOT NULL CHECK (amount_sgd >= 0)
);
```

In [ ]:
result = query("""
SELECT 
    plan,
    COUNT(*) AS user_count
FROM users
GROUP BY plan
ORDER BY user_count DESC
""")
print(result)

---

# Chapter 33: SELECT and WHERE — Reading Data from a Database

## 0. Where You Are

## 1. The Concept

```sql
SELECT column1, column2
FROM table_name
WHERE condition;
```

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### Basic SELECT

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",         "cinemastream/data/users.csv"),
    ("movies",        "cinemastream/data/movies.csv"),
    ("watch_events",  "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

# Select specific columns
result = q("""
SELECT name, country, plan
FROM   users
LIMIT  5
""")
print(result)

In [ ]:
# SELECT * — all columns (for exploration only)
result = q("SELECT * FROM users LIMIT 3")
print(result.columns.tolist())

### Column Aliases with AS

In [ ]:
result = q("""
SELECT 
    name               AS user_name,
    plan               AS subscription_plan,
    country            AS registered_country
FROM users
LIMIT 5
""")
print(result)

### WHERE — Comparison Operators

In [ ]:
# Equality
premium_users = q("""
SELECT user_id, name, plan
FROM   users
WHERE  plan = 'Premium'
""")
print(f"Premium users: {len(premium_users)}")
print(premium_users.head(3))

In [ ]:
# Greater than / less than
long_sessions = q("""
SELECT event_id, user_id, watch_minutes
FROM   watch_events
WHERE  watch_minutes > 120
LIMIT  5
""")
print(long_sessions)

### WHERE — AND, OR, NOT

In [ ]:
# AND: multiple conditions must be true
sg_premium = q("""
SELECT user_id, name, country, plan
FROM   users
WHERE  country = 'SG'
  AND  plan = 'Premium'
""")
print("Singapore Premium users:")
print(sg_premium)

In [ ]:
# OR: at least one condition must be true
sg_or_my = q("""
SELECT user_id, name, country, plan
FROM   users
WHERE  country = 'SG'
   OR  country = 'MY'
ORDER BY country, user_id
LIMIT  5
""")
print(sg_or_my)

In [ ]:
# NOT
non_premium = q("""
SELECT COUNT(*) AS count
FROM   users
WHERE  NOT plan = 'Premium'
""")
print(f"Non-premium users: {non_premium['count'][0]}")

### WHERE — BETWEEN, IN, LIKE, IS NULL

In [ ]:
# BETWEEN (inclusive)
moderate_watchers = q("""
SELECT event_id, watch_minutes
FROM   watch_events
WHERE  watch_minutes BETWEEN 60 AND 90
LIMIT  5
""")
print("Sessions 60–90 minutes:")
print(moderate_watchers)

In [ ]:
# IN — membership test (cleaner than multiple OR conditions)
high_value_countries = q("""
SELECT user_id, name, country, plan
FROM   users
WHERE  country IN ('SG', 'IN', 'MY')
  AND  plan = 'Premium'
ORDER BY country
""")
print("Premium users in high-value markets:")
print(high_value_countries)

In [ ]:
# LIKE — pattern matching (% = any string, _ = one char)
gmail_users = q("""
SELECT user_id, name, email
FROM   users
WHERE  email LIKE '%@example.com'
LIMIT  3
""")
print(gmail_users)

In [ ]:
# IS NULL vs IS NOT NULL
no_name = q("""
SELECT COUNT(*) AS users_without_name
FROM   users
WHERE  name IS NULL
""")
print(f"Users with NULL name: {no_name['users_without_name'][0]}")

### ORDER BY and LIMIT

In [ ]:
# Sort and limit
top_watchers = q("""
SELECT user_id, watch_minutes
FROM   watch_events
WHERE  watch_minutes IS NOT NULL
ORDER BY watch_minutes DESC
LIMIT 5
""")
print(top_watchers)

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Setup (same as above — each script is self-contained)
conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
active_premium = q("""
SELECT COUNT(*) AS active_premium_count
FROM   users
WHERE  plan    = 'Premium'
  AND  churned = 0
""")
print(f"Active Premium users: {active_premium['active_premium_count'][0]}")

In [ ]:
sea_paid = q("""
SELECT 
    user_id,
    name,
    country,
    plan,
    signup_date
FROM   users
WHERE  country IN ('ID', 'PH')
  AND  plan IN ('Basic', 'Premium')
ORDER BY country, plan, user_id
""")
print(f"Indonesian and Filipino paid users: {len(sea_paid)}")
print(sea_paid.head(8))

In [ ]:
suspicious_sessions = q("""
SELECT 
    event_id,
    user_id,
    movie_id,
    watch_minutes,
    device,
    country
FROM   watch_events
WHERE  watch_minutes > 120
ORDER BY watch_minutes DESC
""")
print(f"Sessions > 120 minutes: {len(suspicious_sessions)}")
print(suspicious_sessions)

In [ ]:
new_premium_2023 = q("""
SELECT 
    user_id,
    name,
    country,
    signup_date
FROM   users
WHERE  plan           = 'Premium'
  AND  churned        = 0
  AND  signup_date LIKE '2023-%'
ORDER BY signup_date
""")
print(f"2023 Premium signups (active): {len(new_premium_2023)}")
print(new_premium_2023)

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = q("""
SELECT 
    user_id,
    name,
    signup_date
FROM   users
WHERE  country      = 'VN'
  AND  signup_date  < '2023-01-01'
ORDER BY signup_date
""")
print(result)

```sql
SELECT user_id, name 
FROM users 
WHERE name = NULL
```

In [ ]:
result = q("""
SELECT user_id, name 
FROM   users 
WHERE  name IS NULL
LIMIT  5
""")
print(f"Users with no name: {len(result)}")
print(result)

In [ ]:
result = q("""
SELECT 
    event_id,
    user_id,
    watch_minutes,
    device
FROM   watch_events
WHERE  completed     = 1
  AND  device        = 'Mobile'
  AND  watch_minutes BETWEEN 45 AND 90
ORDER BY watch_minutes DESC
""")
print(f"Completed mobile sessions (45–90 min): {len(result)}")
print(result.head(5))

---

# Chapter 34: GROUP BY and HAVING — Aggregation in SQL

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### Basic GROUP BY with COUNT

In [ ]:
# Count users per plan
result = q("""
SELECT 
    plan,
    COUNT(*) AS user_count
FROM   users
GROUP BY plan
ORDER BY user_count DESC
""")
print(result)

### Multiple Aggregate Functions

In [ ]:
result = q("""
SELECT 
    device,
    COUNT(*)           AS total_sessions,
    ROUND(AVG(watch_minutes), 1) AS avg_minutes,
    SUM(watch_minutes) AS total_minutes,
    MAX(watch_minutes) AS longest_session
FROM   watch_events
WHERE  watch_minutes IS NOT NULL
GROUP BY device
ORDER BY total_sessions DESC
""")
print(result)

### GROUP BY Multiple Columns

In [ ]:
# Group by two columns: plan × country
result = q("""
SELECT 
    country,
    plan,
    COUNT(*)  AS user_count
FROM   users
GROUP BY country, plan
ORDER BY country, user_count DESC
""")
print(result)

### HAVING — Filter on Aggregated Values

In [ ]:
# Countries where average session is over 75 minutes
high_engagement = q("""
SELECT 
    country,
    COUNT(*)                     AS session_count,
    ROUND(AVG(watch_minutes), 1) AS avg_minutes
FROM   watch_events
WHERE  watch_minutes IS NOT NULL
GROUP BY country
HAVING AVG(watch_minutes) > 75
ORDER BY avg_minutes DESC
""")
print("High-engagement countries (avg > 75 min):")
print(high_engagement)

In [ ]:
# WHERE filters rows before GROUP BY; HAVING filters after
# Compare:
result1 = q("""
SELECT plan, COUNT(*) AS n
FROM   users
WHERE  country = 'MY'     -- filter rows BEFORE grouping
GROUP BY plan
""")

result2 = q("""
SELECT plan, COUNT(*) AS n
FROM   users
GROUP BY plan
HAVING COUNT(*) > 20     -- filter GROUPS after grouping
""")

print("Plans in Malaysia:")
print(result1)
print("\nPlans with more than 20 users (globally):")
print(result2)

### COUNT(*) vs COUNT(column)

In [ ]:
# COUNT(*) counts all rows in the group
# COUNT(column) counts non-NULL values only
result = q("""
SELECT
    COUNT(*)     AS total_rows,
    COUNT(name)  AS rows_with_name,
    COUNT(*)     - COUNT(name) AS missing_names
FROM users
""")
print(result)

### Combining WHERE and HAVING

In [ ]:
# WHERE: only non-churned users
# HAVING: only countries with more than 5 non-churned users
result = q("""
SELECT 
    country,
    COUNT(*)            AS active_user_count
FROM   users
WHERE  churned = 0
GROUP BY country
HAVING COUNT(*) > 8
ORDER BY active_user_count DESC
""")
print("Countries with >8 active users:")
print(result)

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
plan_health = q("""
SELECT
    plan,
    COUNT(*)                                      AS total,
    SUM(churned)                                  AS churned,
    ROUND(100.0 * SUM(churned) / COUNT(*), 1)     AS churn_rate_pct
FROM   users
GROUP BY plan
ORDER BY total DESC
""")
print("=== Plan Health ===")
print(plan_health.to_string(index=False))

In [ ]:
engagement = q("""
SELECT
    we.country,
    COUNT(*)                            AS sessions,
    COUNT(DISTINCT we.user_id)          AS unique_viewers,
    ROUND(1.0 * COUNT(*) / COUNT(DISTINCT we.user_id), 1) AS sessions_per_viewer,
    ROUND(AVG(we.watch_minutes), 1)     AS avg_session_min
FROM   watch_events we
WHERE  we.watch_minutes IS NOT NULL
GROUP BY we.country
ORDER BY sessions_per_viewer DESC
""")
print("=== Engagement by Country ===")
print(engagement.to_string(index=False))

In [ ]:
top_movies = q("""
SELECT
    m.title,
    m.genre,
    COUNT(*)                                           AS watch_count,
    ROUND(100.0 * SUM(we.completed) / COUNT(*), 1)    AS completion_pct
FROM   watch_events we
JOIN   movies m ON we.movie_id = m.movie_id
GROUP BY m.movie_id, m.title, m.genre
HAVING COUNT(*) >= 8
ORDER BY watch_count DESC, completion_pct DESC
LIMIT 5
""")
print("=== Top Movies (8+ watches) ===")
print(top_movies.to_string(index=False))

In [ ]:
orphan_check = q("""
SELECT 
    COUNT(DISTINCT we.user_id)  AS event_user_ids,
    COUNT(DISTINCT u.user_id)   AS user_table_ids
FROM      watch_events we
LEFT JOIN users u ON we.user_id = u.user_id
WHERE     u.user_id IS NULL
""")
print("=== Orphaned Watch Events ===")
print(orphan_check)

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = q("""
SELECT 
    device,
    COUNT(*) AS completed_sessions
FROM   watch_events
WHERE  completed = 1
GROUP BY device
ORDER BY completed_sessions DESC
""")
print(result)

```sql
SELECT country, AVG(watch_minutes) AS avg_min
FROM   watch_events
WHERE  AVG(watch_minutes) > 75
GROUP BY country
```

In [ ]:
result = q("""
SELECT country, ROUND(AVG(watch_minutes), 1) AS avg_min
FROM   watch_events
GROUP BY country
HAVING AVG(watch_minutes) > 75
ORDER BY avg_min DESC
""")
print(result)

In [ ]:
result = q("""
SELECT 
    country,
    COUNT(DISTINCT movie_id)                         AS distinct_movies,
    ROUND(100.0 * SUM(completed) / COUNT(*), 1)      AS completion_pct
FROM   watch_events
GROUP BY country
HAVING COUNT(DISTINCT movie_id) >= 5
ORDER BY completion_pct DESC
""")
print(result)

---

# Chapter 35: SQL JOINs — Combining Tables

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### INNER JOIN

In [ ]:
# Get user names alongside their watch events
inner_result = q("""
SELECT
    we.event_id,
    u.name,
    u.plan,
    we.watch_minutes,
    we.device
FROM   watch_events we
INNER JOIN users u ON we.user_id = u.user_id
LIMIT 5
""")
print("INNER JOIN — events with user details:")
print(inner_result)

### LEFT JOIN — Keep All Left Rows

In [ ]:
# All users, even those with no watch events
all_users_events = q("""
SELECT
    u.user_id,
    u.name,
    u.plan,
    COUNT(we.event_id) AS event_count
FROM      users u
LEFT JOIN watch_events we ON u.user_id = we.user_id
GROUP BY  u.user_id, u.name, u.plan
ORDER BY  event_count DESC
LIMIT 8
""")
print("Users with event counts (including 0-event users):")
print(all_users_events)

In [ ]:
# Find users with NO events — the inactive cohort
inactive_users = q("""
SELECT
    u.user_id,
    u.name,
    u.plan,
    u.country
FROM      users u
LEFT JOIN watch_events we ON u.user_id = we.user_id
WHERE     we.event_id IS NULL
""")
print(f"\nInactive users (no watch events): {len(inactive_users)}")
print(inactive_users.head(5))

### JOIN Three Tables

In [ ]:
# Watch events with user details AND movie details
three_table = q("""
SELECT
    we.event_id,
    u.name         AS viewer,
    u.plan,
    m.title        AS movie,
    m.genre,
    we.watch_minutes,
    we.completed
FROM   watch_events we
JOIN   users  u ON we.user_id  = u.user_id
JOIN   movies m ON we.movie_id = m.movie_id
LIMIT 6
""")
print("Three-table JOIN:")
print(three_table)

### JOIN with GROUP BY

In [ ]:
# Average watch minutes per genre, from the three-table join
genre_avg = q("""
SELECT
    m.genre,
    COUNT(*)                            AS sessions,
    ROUND(AVG(we.watch_minutes), 1)     AS avg_minutes,
    ROUND(100.0 * SUM(we.completed) / COUNT(*), 1) AS completion_pct
FROM   watch_events we
JOIN   movies m ON we.movie_id = m.movie_id
GROUP BY m.genre
ORDER BY avg_minutes DESC
""")
print("Genre engagement stats:")
print(genre_avg)

### Detecting Join Cardinality Issues

In [ ]:
# Before a join, always check: is the join key unique on one side?
user_id_dupes = q("""
SELECT user_id, COUNT(*) AS cnt
FROM   users
GROUP BY user_id
HAVING COUNT(*) > 1
""")
print(f"Duplicate user_ids in users table: {len(user_id_dupes)}")

In [ ]:
# If duplicates exist on BOTH sides, the join multiplies rows
# (e.g., 3 events for user 1 × 1 user row for user 1 = 3 output rows — correct)
# (e.g., 3 events for user 1 × 3 user rows for user 1 = 9 output rows — WRONG)
event_count_pre  = q("SELECT COUNT(*) AS n FROM watch_events")['n'][0]
event_count_post = q("""
    SELECT COUNT(*) AS n 
    FROM watch_events we 
    JOIN users u ON we.user_id = u.user_id
""")['n'][0]

print(f"Events before join: {event_count_pre}")
print(f"Events after inner join: {event_count_post}")
print(f"Row count preserved: {event_count_pre == event_count_post}")

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
top_viewers = q("""
SELECT
    u.name,
    u.country,
    COUNT(we.event_id)              AS sessions,
    SUM(we.watch_minutes)           AS total_minutes,
    ROUND(AVG(we.watch_minutes), 1) AS avg_minutes
FROM      users u
JOIN      watch_events we ON u.user_id = we.user_id
WHERE     u.plan    = 'Premium'
  AND     u.churned = 0
GROUP BY  u.user_id, u.name, u.country
ORDER BY  total_minutes DESC
LIMIT 5
""")
print("=== Top Premium Viewers ===")
print(top_viewers.to_string(index=False))

In [ ]:
movie_abandon = q("""
SELECT
    m.title,
    m.genre,
    m.runtime_min,
    COUNT(we.event_id)                              AS total_watches,
    SUM(CASE WHEN we.completed = 0 THEN 1 ELSE 0 END) AS abandoned,
    ROUND(100.0 * SUM(CASE WHEN we.completed = 0 THEN 1 ELSE 0 END) / COUNT(*), 1) AS abandon_pct
FROM   movies m
JOIN   watch_events we ON m.movie_id = we.movie_id
GROUP BY m.movie_id, m.title, m.genre, m.runtime_min
HAVING COUNT(we.event_id) >= 5
ORDER BY abandon_pct DESC
LIMIT 5
""")
print("\n=== Most Abandoned Movies (5+ watches) ===")
print(movie_abandon.to_string(index=False))

In [ ]:
inactive_paid = q("""
SELECT
    u.user_id,
    u.name,
    u.plan,
    u.country,
    u.signup_date
FROM      users u
LEFT JOIN watch_events we ON u.user_id = we.user_id
WHERE     u.plan IN ('Basic', 'Premium')
  AND     u.churned = 0
  AND     we.event_id IS NULL
ORDER BY  u.plan DESC, u.signup_date
""")
print(f"\n=== Inactive Paying Users: {len(inactive_paid)} ===")
print(inactive_paid.head(6).to_string(index=False))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = q("""
SELECT
    m.title,
    m.genre,
    COUNT(we.event_id) AS watch_count
FROM   movies m
JOIN   watch_events we ON m.movie_id = we.movie_id
GROUP BY m.movie_id, m.title, m.genre
ORDER BY watch_count DESC
LIMIT 8
""")
print(result)

In [ ]:
result = q("""
SELECT
    m.movie_id,
    m.title,
    m.genre
FROM      movies m
LEFT JOIN watch_events we ON m.movie_id = we.movie_id
WHERE     we.event_id IS NULL
""")
print(f"Movies never watched: {len(result)}")
print(result)

In [ ]:
genre_by_plan = q("""
SELECT
    u.plan,
    m.genre,
    COUNT(*) AS watches
FROM   watch_events we
JOIN   users  u ON we.user_id  = u.user_id
JOIN   movies m ON we.movie_id = m.movie_id
GROUP BY u.plan, m.genre
ORDER BY u.plan, watches DESC
""")
print("Genre popularity by plan:")
print(genre_by_plan.to_string(index=False))

---

# Chapter 36: Subqueries and CTEs — Queries Inside Queries

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### Subquery in WHERE — IN

In [ ]:
# Find users who have watched at least one Drama
drama_watchers = q("""
SELECT user_id, name, plan
FROM   users
WHERE  user_id IN (
    SELECT DISTINCT we.user_id
    FROM   watch_events we
    JOIN   movies m ON we.movie_id = m.movie_id
    WHERE  m.genre = 'Drama'
)
ORDER BY user_id
LIMIT 6
""")
print("Users who watched Drama:")
print(drama_watchers)

### Subquery in WHERE — Scalar Comparison

In [ ]:
# Users whose total watch time is above the average user's total watch time
above_avg_viewers = q("""
SELECT 
    user_id,
    SUM(watch_minutes) AS total_minutes
FROM   watch_events
GROUP BY user_id
HAVING SUM(watch_minutes) > (
    SELECT AVG(user_total)
    FROM (
        SELECT SUM(watch_minutes) AS user_total
        FROM   watch_events
        GROUP BY user_id
    )
)
ORDER BY total_minutes DESC
LIMIT 6
""")
print("Above-average viewers by total minutes:")
print(above_avg_viewers)

### Subquery in FROM — Derived Table

In [ ]:
# Average sessions per user by country, then compare countries
sessions_per_country = q("""
SELECT 
    country,
    ROUND(AVG(user_sessions), 1) AS avg_sessions_per_user
FROM (
    SELECT 
        country,
        user_id,
        COUNT(*) AS user_sessions
    FROM   watch_events
    GROUP BY country, user_id
) AS user_session_counts
GROUP BY country
ORDER BY avg_sessions_per_user DESC
""")
print("Avg sessions per user by country:")
print(sessions_per_country)

### CTEs — The Readable Way

In [ ]:
# Rewrite the above as a CTE — same result, much clearer
sessions_per_country_cte = q("""
WITH user_session_counts AS (
    SELECT 
        country,
        user_id,
        COUNT(*) AS user_sessions
    FROM   watch_events
    GROUP BY country, user_id
)
SELECT 
    country,
    ROUND(AVG(user_sessions), 1) AS avg_sessions_per_user
FROM   user_session_counts
GROUP BY country
ORDER BY avg_sessions_per_user DESC
""")
print("Same result via CTE:")
print(sessions_per_country_cte)

### Multiple CTEs

In [ ]:
multi_cte = q("""
WITH 
-- CTE 1: active non-churned users
active_users AS (
    SELECT user_id, name, plan, country
    FROM   users
    WHERE  churned = 0
),
-- CTE 2: their total watch time
user_totals AS (
    SELECT 
        user_id,
        SUM(watch_minutes) AS total_minutes,
        COUNT(*)            AS session_count
    FROM   watch_events
    GROUP BY user_id
)
-- Main query: join both CTEs
SELECT
    au.name,
    au.plan,
    au.country,
    COALESCE(ut.session_count, 0)   AS sessions,
    COALESCE(ut.total_minutes, 0)   AS total_minutes
FROM      active_users au
LEFT JOIN user_totals  ut ON au.user_id = ut.user_id
ORDER BY  total_minutes DESC
LIMIT 6
""")
print("Active users ranked by watch time:")
print(multi_cte)

### COALESCE — NULL Replacement

In [ ]:
coalesce_demo = q("""
SELECT
    user_id,
    name,
    COALESCE(name, 'Unknown User') AS display_name
FROM users
WHERE name IS NULL
LIMIT 4
""")
print(coalesce_demo)

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
engaged_users = q("""
WITH user_stats AS (
    SELECT
        user_id,
        COUNT(*)                                         AS sessions,
        SUM(watch_minutes)                               AS total_minutes,
        ROUND(100.0 * SUM(completed) / COUNT(*), 1)     AS completion_rate
    FROM watch_events
    WHERE watch_minutes IS NOT NULL
    GROUP BY user_id
),
-- Compute the benchmark averages
benchmarks AS (
    SELECT
        AVG(total_minutes)   AS avg_total_minutes,
        AVG(completion_rate) AS avg_completion_rate
    FROM user_stats
)
-- Select users above both benchmarks
SELECT
    us.user_id,
    u.name,
    u.plan,
    u.country,
    us.sessions,
    us.total_minutes,
    us.completion_rate
FROM      user_stats  us
JOIN      users       u  ON us.user_id = u.user_id
CROSS JOIN benchmarks bm
WHERE     us.total_minutes   > bm.avg_total_minutes
  AND     us.completion_rate > bm.avg_completion_rate
ORDER BY  us.total_minutes DESC
""")
print(f"High-engagement users (above average on both metrics): {len(engaged_users)}")
print(engaged_users)

In [ ]:
# Now add movie preference — top genre per high-engagement user
top_genre_per_user = q("""
WITH user_stats AS (
    SELECT
        user_id,
        SUM(watch_minutes)                               AS total_minutes,
        ROUND(100.0 * SUM(completed) / COUNT(*), 1)     AS completion_rate
    FROM watch_events
    GROUP BY user_id
),
benchmarks AS (
    SELECT AVG(total_minutes) AS avg_min, AVG(completion_rate) AS avg_rate
    FROM user_stats
),
high_engagement AS (
    SELECT us.user_id
    FROM   user_stats us
    CROSS JOIN benchmarks bm
    WHERE  us.total_minutes   > bm.avg_min
      AND  us.completion_rate > bm.avg_rate
),
genre_counts AS (
    SELECT
        we.user_id,
        m.genre,
        COUNT(*) AS genre_watches
    FROM   watch_events we
    JOIN   movies m ON we.movie_id = m.movie_id
    WHERE  we.user_id IN (SELECT user_id FROM high_engagement)
    GROUP BY we.user_id, m.genre
)
SELECT
    user_id,
    genre           AS top_genre,
    genre_watches
FROM (
    SELECT
        user_id,
        genre,
        genre_watches,
        RANK() OVER (PARTITION BY user_id ORDER BY genre_watches DESC) AS rnk
    FROM genre_counts
)
WHERE rnk = 1
ORDER BY user_id
LIMIT 8
""")
print("\nTop genre for high-engagement users:")
print(top_genre_per_user)

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = q("""
WITH movie_avg AS (
    SELECT
        movie_id,
        AVG(watch_minutes) AS avg_minutes
    FROM watch_events
    GROUP BY movie_id
),
global_avg AS (
    SELECT AVG(watch_minutes) AS overall_avg
    FROM   watch_events
)
SELECT
    m.title,
    m.genre,
    ROUND(ma.avg_minutes, 1) AS avg_watch_min,
    ROUND(ga.overall_avg, 1) AS global_avg
FROM      movie_avg  ma
JOIN      movies     m  ON ma.movie_id = m.movie_id
CROSS JOIN global_avg ga
WHERE     ma.avg_minutes > ga.overall_avg
ORDER BY  ma.avg_minutes DESC
LIMIT 5
""")
print(result)

```sql
SELECT country, ROUND(AVG(session_count), 1) AS avg_sessions
FROM (
    SELECT country, user_id, COUNT(*) AS session_count
    FROM   watch_events
    GROUP BY country, user_id
) sub
GROUP BY country
ORDER BY avg_sessions DESC
```

In [ ]:
result = q("""
WITH sessions_per_user_country AS (
    SELECT 
        country,
        user_id,
        COUNT(*) AS session_count
    FROM   watch_events
    GROUP BY country, user_id
)
SELECT 
    country,
    ROUND(AVG(session_count), 1) AS avg_sessions
FROM   sessions_per_user_country
GROUP BY country
ORDER BY avg_sessions DESC
""")
print(result)

In [ ]:
result = q("""
WITH device_variety AS (
    SELECT 
        user_id,
        COUNT(DISTINCT device) AS devices_used,
        MAX(CASE WHEN device = 'Mobile' THEN 1 ELSE 0 END) AS has_mobile
    FROM   watch_events
    GROUP BY user_id
),
mobile_only AS (
    SELECT user_id
    FROM   device_variety
    WHERE  devices_used = 1
      AND  has_mobile   = 1
)
SELECT
    u.user_id,
    u.name,
    u.plan,
    u.country
FROM      mobile_only mo
JOIN      users u ON mo.user_id = u.user_id
ORDER BY  u.country, u.user_id
""")
print(f"Mobile-only watchers: {len(result)}")
print(result)

---

# Chapter 37: Window Functions — Aggregations Without Collapsing Rows

## 0. Where You Are

## 1. The Concept

```sql
function_name() OVER (
    PARTITION BY partition_column  -- like GROUP BY: separate window per group
    ORDER BY     order_column      -- defines row order within the window
    ROWS BETWEEN ...               -- optional: further restrict window frame
)
```

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### ROW_NUMBER, RANK, DENSE_RANK

In [ ]:
ranking_demo = q("""
SELECT
    user_id,
    watch_minutes,
    ROW_NUMBER() OVER (ORDER BY watch_minutes DESC) AS row_num,
    RANK()       OVER (ORDER BY watch_minutes DESC) AS rank_val,
    DENSE_RANK() OVER (ORDER BY watch_minutes DESC) AS dense_rank_val
FROM watch_events
LIMIT 8
""")
print("Ranking by watch_minutes:")
print(ranking_demo)

In [ ]:
# With PARTITION BY — rank WITHIN each plan tier
ranked_by_plan = q("""
SELECT
    we.user_id,
    u.plan,
    SUM(we.watch_minutes)                                  AS total_min,
    RANK() OVER (PARTITION BY u.plan ORDER BY SUM(we.watch_minutes) DESC) AS plan_rank
FROM   watch_events we
JOIN   users u ON we.user_id = u.user_id
GROUP BY we.user_id, u.plan
ORDER BY u.plan, plan_rank
LIMIT 9
""")
print("\nTop viewers ranked within each plan:")
print(ranked_by_plan)

### SUM OVER — Running Total

In [ ]:
running_total = q("""
SELECT
    event_id,
    watch_minutes,
    SUM(watch_minutes) OVER (ORDER BY event_id) AS running_total_minutes
FROM watch_events
WHERE user_id = 1
ORDER BY event_id
""")
print("Running total watch time for user 1:")
print(running_total)

### Aggregate OVER PARTITION — Group Total on Each Row

In [ ]:
# For each event, show that user's total vs country total
comparison = q("""
SELECT
    we.event_id,
    we.user_id,
    we.watch_minutes,
    SUM(we.watch_minutes) OVER (PARTITION BY we.user_id)   AS user_total,
    SUM(we.watch_minutes) OVER (PARTITION BY we.country)   AS country_total,
    ROUND(100.0 * we.watch_minutes /
        SUM(we.watch_minutes) OVER (PARTITION BY we.country), 1) AS pct_of_country
FROM watch_events we
WHERE we.country = 'SG'
  AND we.user_id IN (1, 7, 8)
LIMIT 6
""")
print("Contribution to country total:")
print(comparison)

### LAG and LEAD — Row Offset Values

In [ ]:
# LAG: compare each session's duration to the previous session
lag_demo = q("""
SELECT
    event_id,
    watch_started,
    watch_minutes,
    LAG(watch_minutes, 1, 0) OVER (PARTITION BY user_id ORDER BY watch_started) AS prev_session,
    watch_minutes - LAG(watch_minutes, 1, 0) OVER (PARTITION BY user_id ORDER BY watch_started) AS delta
FROM watch_events
WHERE user_id = 1
ORDER BY watch_started
""")
print("Session-to-session comparison for user 1:")
print(lag_demo)

### NTILE — Bucket Rows into Percentile Groups

In [ ]:
# Divide users into 4 quartiles by total watch time
quartiles = q("""
WITH user_totals AS (
    SELECT 
        user_id,
        SUM(watch_minutes) AS total_minutes
    FROM   watch_events
    GROUP BY user_id
)
SELECT
    user_id,
    total_minutes,
    NTILE(4) OVER (ORDER BY total_minutes DESC) AS quartile
FROM user_totals
ORDER BY quartile, total_minutes DESC
LIMIT 8
""")
print("Watch-time quartiles:")
print(quartiles)

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
user_ranking = q("""
WITH user_stats AS (
    SELECT
        we.user_id,
        u.plan,
        u.country,
        SUM(we.watch_minutes)  AS total_minutes,
        COUNT(we.event_id)     AS sessions
    FROM   watch_events we
    JOIN   users u ON we.user_id = u.user_id
    GROUP BY we.user_id, u.plan, u.country
)
SELECT
    user_id,
    plan,
    country,
    total_minutes,
    sessions,
    RANK() OVER (PARTITION BY plan ORDER BY total_minutes DESC)    AS plan_rank,
    RANK() OVER (PARTITION BY country ORDER BY total_minutes DESC) AS country_rank
FROM user_stats
WHERE plan = 'Premium'
ORDER BY total_minutes DESC
""")
print("Premium user rankings:")
print(user_ranking)

In [ ]:
top_movie_by_country = q("""
WITH country_movie_counts AS (
    SELECT
        we.country,
        m.title,
        m.genre,
        COUNT(*) AS watch_count,
        RANK() OVER (PARTITION BY we.country ORDER BY COUNT(*) DESC) AS rnk
    FROM   watch_events we
    JOIN   movies m ON we.movie_id = m.movie_id
    GROUP BY we.country, m.movie_id, m.title, m.genre
)
SELECT country, title, genre, watch_count
FROM   country_movie_counts
WHERE  rnk = 1
ORDER BY country
""")
print("\nMost watched movie per country:")
print(top_movie_by_country)

In [ ]:
vs_plan_avg = q("""
WITH user_stats AS (
    SELECT
        we.user_id,
        u.plan,
        SUM(we.watch_minutes) AS total_minutes
    FROM   watch_events we
    JOIN   users u ON we.user_id = u.user_id
    GROUP BY we.user_id, u.plan
)
SELECT
    user_id,
    plan,
    total_minutes,
    ROUND(AVG(total_minutes) OVER (PARTITION BY plan), 1) AS plan_avg,
    ROUND(total_minutes - AVG(total_minutes) OVER (PARTITION BY plan), 1) AS vs_plan_avg,
    CASE 
        WHEN total_minutes > AVG(total_minutes) OVER (PARTITION BY plan) 
        THEN 'above_avg'
        ELSE 'below_avg'
    END AS segment
FROM user_stats
ORDER BY plan, total_minutes DESC
LIMIT 8
""")
print("\nUsers vs plan average:")
print(vs_plan_avg)

## 4. Pitfalls & Pro Tips

## 5. Exercises

```sql
SELECT 
    plan,
    COUNT(*) AS n,
    SUM(COUNT(*)) OVER () AS total
FROM users
GROUP BY plan
```

In [ ]:
result = q("""
SELECT 
    plan,
    COUNT(*) AS n,
    SUM(COUNT(*)) OVER () AS total
FROM users
GROUP BY plan
""")
print(result)

In [ ]:
result = q("""
WITH ranked_events AS (
    SELECT
        country,
        event_id,
        watch_minutes,
        RANK() OVER (PARTITION BY country ORDER BY watch_minutes DESC) AS rnk
    FROM watch_events
    WHERE watch_minutes IS NOT NULL
)
SELECT country, event_id, watch_minutes, rnk
FROM   ranked_events
WHERE  rnk <= 3
ORDER BY country, rnk
""")
print(result.head(12))

In [ ]:
result = q("""
WITH user_sessions AS (
    SELECT
        user_id,
        event_id,
        watch_started,
        watch_minutes,
        LAG(watch_minutes, 1) OVER (PARTITION BY user_id ORDER BY watch_started) AS prev_session_min,
        MAX(watch_minutes)    OVER (PARTITION BY user_id)                         AS personal_best
    FROM watch_events
    WHERE watch_minutes IS NOT NULL
)
SELECT
    user_id,
    event_id,
    watch_minutes,
    prev_session_min,
    personal_best,
    watch_minutes - COALESCE(prev_session_min, watch_minutes) AS delta_from_prev
FROM user_sessions
WHERE user_id IN (1, 2)
ORDER BY user_id, watch_started
""")
print(result)

---

# Chapter 38: Writing Queries for Real Datasets — Putting It All Together

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### Technique 1: Build Complex Queries Incrementally

In [ ]:
# Question: "Who are the top 5 users by revenue contribution per country?"
# Step 1: compute user monthly revenue
# Step 2: rank within country
# Step 3: filter to top 5 per country

# Step 1 — build and test alone
user_revenue = q("""
SELECT
    u.user_id,
    u.country,
    u.plan,
    CASE u.plan
        WHEN 'Premium' THEN 12.90
        WHEN 'Basic'   THEN  8.90
        ELSE 0.0
    END AS monthly_sgd
FROM   users u
WHERE  u.churned = 0
LIMIT 5
""")
print("Step 1 (revenue per user):")
print(user_revenue)

In [ ]:
# Step 2 — add ranking; verify step 1 first
top_per_country = q("""
WITH user_revenue AS (
    SELECT
        user_id,
        country,
        plan,
        CASE plan
            WHEN 'Premium' THEN 12.90
            WHEN 'Basic'   THEN  8.90
            ELSE 0.0
        END AS monthly_sgd
    FROM   users
    WHERE  churned = 0
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY country ORDER BY monthly_sgd DESC, user_id) AS country_rank
    FROM user_revenue
)
SELECT country, user_id, plan, monthly_sgd, country_rank
FROM   ranked
WHERE  country_rank <= 2
ORDER BY country, country_rank
""")
print("\nTop 2 revenue users per country:")
print(top_per_country)

### Technique 2: Defensive NULL Handling

In [ ]:
# Always count NULLs before aggregating
null_audit = q("""
SELECT
    SUM(CASE WHEN watch_minutes IS NULL THEN 1 ELSE 0 END) AS null_minutes,
    SUM(CASE WHEN completed IS NULL THEN 1 ELSE 0 END)     AS null_completed,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END)       AS null_user_id,
    COUNT(*) AS total_rows
FROM watch_events
""")
print("NULL audit on watch_events:")
print(null_audit)

In [ ]:
# Defensive COALESCE in aggregation — protect against NULL contamination
safe_avg = q("""
SELECT
    device,
    AVG(watch_minutes)                         AS avg_raw,
    AVG(COALESCE(watch_minutes, 0))            AS avg_nulls_as_zero,  -- wrong for avg!
    AVG(NULLIF(watch_minutes, 0))              AS avg_zeros_excluded   -- correct
FROM watch_events
GROUP BY device
ORDER BY avg_raw DESC
""")
print("\nDefensive averaging comparison:")
print(safe_avg)

### Technique 3: Deduplication with ROW_NUMBER

In [ ]:
# Simulate a duplicated dataset (as arrives from ETL)
import pandas as pd
import io

dup_data = """user_id,plan,signup_date
1,Premium,2022-01-15
1,Basic,2022-01-15
2,Free,2022-03-10
3,Basic,2022-05-01
3,Basic,2022-05-01"""

dup_df = pd.read_csv(io.StringIO(dup_data))
dup_df.to_sql("users_raw", conn, index=False, if_exists="replace")

# Dedup: keep latest record per user_id by ROW_NUMBER
deduped = q("""
WITH deduped AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY signup_date DESC) AS rn
    FROM users_raw
)
SELECT user_id, plan, signup_date
FROM   deduped
WHERE  rn = 1
""")
print("Deduplicated users:")
print(deduped)

### Technique 4: Cross-Database Sanity Checks

In [ ]:
# After any aggregation, verify against known totals
plan_sums = q("""
SELECT 
    SUM(CASE WHEN plan = 'Free' THEN 1 ELSE 0 END) AS free_count,
    SUM(CASE WHEN plan = 'Basic' THEN 1 ELSE 0 END) AS basic_count,
    SUM(CASE WHEN plan = 'Premium' THEN 1 ELSE 0 END) AS premium_count,
    COUNT(*) AS total
FROM users
""")
print("Sanity check — plan counts vs total:")
print(plan_sums)
row = plan_sums.iloc[0]
assert row['free_count'] + row['basic_count'] + row['premium_count'] == row['total'], \
    "Plan counts don't sum to total!"
print("Assertion passed.")

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
weekly_review = q("""
WITH

-- Metric 1: Subscriber health
subscriber_health AS (
    SELECT
        plan,
        COUNT(*)                                          AS total,
        SUM(churned)                                      AS churned,
        ROUND(100.0 * SUM(churned) / COUNT(*), 1)        AS churn_rate_pct,
        COUNT(*) - SUM(churned)                           AS active
    FROM users
    GROUP BY plan
),

-- Metric 2: Monthly revenue (active subscribers only)
monthly_revenue AS (
    SELECT
        SUM(CASE plan
            WHEN 'Premium' THEN 12.90
            WHEN 'Basic'   THEN  8.90
            ELSE 0.0
        END)  AS total_mrr_sgd
    FROM users
    WHERE churned = 0
),

-- Metric 3: Watch engagement
watch_summary AS (
    SELECT
        COUNT(DISTINCT user_id)             AS active_viewers,
        COUNT(*)                            AS total_sessions,
        ROUND(AVG(watch_minutes), 1)        AS avg_session_min,
        ROUND(100.0 * SUM(completed) / COUNT(*), 1) AS completion_pct
    FROM watch_events
),

-- Metric 4: Top content
top_movie AS (
    SELECT
        m.title,
        COUNT(*) AS watches
    FROM   watch_events we
    JOIN   movies m ON we.movie_id = m.movie_id
    GROUP BY m.movie_id, m.title
    ORDER BY watches DESC
    LIMIT 1
),

-- Metric 5: Inactive paying users (revenue at risk)
inactive_paid AS (
    SELECT COUNT(*) AS count
    FROM      users u
    LEFT JOIN watch_events we ON u.user_id = we.user_id
    WHERE     u.plan IN ('Basic', 'Premium')
      AND     u.churned = 0
      AND     we.event_id IS NULL
)

-- Assemble all metrics into one readable result
SELECT
    'Subscriber Health'  AS metric_group,
    'Active Premium'     AS metric,
    CAST((SELECT active FROM subscriber_health WHERE plan = 'Premium') AS TEXT) AS value
UNION ALL SELECT 'Subscriber Health', 'Active Basic',
    CAST((SELECT active FROM subscriber_health WHERE plan = 'Basic') AS TEXT)
UNION ALL SELECT 'Subscriber Health', 'Active Free',
    CAST((SELECT active FROM subscriber_health WHERE plan = 'Free') AS TEXT)
UNION ALL SELECT 'Revenue', 'Monthly MRR (S$)',
    CAST((SELECT ROUND(total_mrr_sgd, 2) FROM monthly_revenue) AS TEXT)
UNION ALL SELECT 'Engagement', 'Active Viewers',
    CAST((SELECT active_viewers FROM watch_summary) AS TEXT)
UNION ALL SELECT 'Engagement', 'Avg Session (min)',
    CAST((SELECT avg_session_min FROM watch_summary) AS TEXT)
UNION ALL SELECT 'Engagement', 'Completion Rate (%)',
    CAST((SELECT completion_pct FROM watch_summary) AS TEXT)
UNION ALL SELECT 'Content', 'Top Movie',
    (SELECT title FROM top_movie)
UNION ALL SELECT 'Risk', 'Inactive Paying Users',
    CAST((SELECT count FROM inactive_paid) AS TEXT)
""")

print("=== CinemaStream Weekly Business Review ===")
print(weekly_review.to_string(index=False))

In [ ]:
# Manual check: 12 active Premium × 12.90 + 21 active Basic × 8.90
manual_mrr = 12 * 12.90 + 21 * 8.90
print(f"Manual MRR: S${manual_mrr:.2f}")
print(f"SQL MRR:    S$341.70")
print(f"Match: {abs(manual_mrr - 341.70) < 0.01}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = q("""
WITH genre_tv AS (
    SELECT
        m.genre,
        COUNT(*)                                               AS total_sessions,
        SUM(CASE WHEN we.device = 'TV' THEN 1 ELSE 0 END)    AS tv_sessions
    FROM   watch_events we
    JOIN   movies m ON we.movie_id = m.movie_id
    GROUP BY m.genre
)
SELECT
    genre,
    total_sessions,
    tv_sessions,
    ROUND(100.0 * tv_sessions / total_sessions, 1) AS tv_pct
FROM genre_tv
ORDER BY tv_pct DESC
""")
print(result)

In [ ]:
# Create duplicate table
q_dup = pd.read_sql("SELECT * FROM watch_events", conn)
dup_events = pd.concat([q_dup, q_dup])
dup_events.to_sql("dup_events", conn, index=False, if_exists="replace")

# Check duplicates
before = q("SELECT COUNT(*) AS n FROM dup_events")['n'][0]

# Deduplicate
deduped = q("""
WITH deduped AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY event_id) AS rn
    FROM dup_events
)
SELECT event_id, user_id, watch_minutes, device, country
FROM   deduped
WHERE  rn = 1
ORDER BY event_id
LIMIT 3
""")
after = q("""
    SELECT COUNT(*) AS n FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY event_id) AS rn
        FROM dup_events
    ) WHERE rn = 1
""")['n'][0]

print(f"Before dedup: {before} rows | After dedup: {after} rows")
print(deduped)

In [ ]:
result = q("""
WITH active_users AS (
    SELECT user_id, plan
    FROM   users
    WHERE  churned = 0
),
event_counts AS (
    SELECT user_id, COUNT(*) AS event_count
    FROM   watch_events
    GROUP BY user_id
),
user_risk AS (
    SELECT
        au.user_id,
        au.plan,
        COALESCE(ec.event_count, 0) AS events,
        CASE au.plan
            WHEN 'Premium' THEN 12.90
            WHEN 'Basic'   THEN  8.90
            ELSE 0.0
        END AS monthly_sgd
    FROM      active_users au
    LEFT JOIN event_counts ec ON au.user_id = ec.user_id
)
SELECT
    plan,
    COUNT(*)                                              AS active_users,
    SUM(CASE WHEN events = 0 THEN 1 ELSE 0 END)          AS zero_event_users,
    ROUND(100.0 * SUM(CASE WHEN events = 0 THEN 1 ELSE 0 END) / COUNT(*), 1)  AS pct_inactive,
    ROUND(SUM(CASE WHEN events = 0 THEN monthly_sgd ELSE 0 END), 2)           AS rev_at_risk_sgd
FROM user_risk
GROUP BY plan
ORDER BY rev_at_risk_sgd DESC
""")
print("=== Churn Risk Report ===")
print(result.to_string(index=False))

---

# Chapter 39: SQL Data Types, Casting, and Date Functions

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### CAST — Explicit Type Conversion

In [ ]:
casting = q("""
SELECT
    '42'         AS text_value,
    CAST('42' AS INTEGER)  AS as_integer,
    CAST('42' AS REAL)     AS as_real,
    CAST(42 AS TEXT)       AS back_to_text
""")
print(casting)

In [ ]:
# Dangerous implicit coercion — SQLite is lenient, PostgreSQL is strict
implicit_coercion = q("""
SELECT
    '10' + 5   AS sqlite_adds_numbers,   -- works in SQLite only
    '10' || 5  AS concatenated           -- || = string concat in SQL
""")
print("Implicit coercion (SQLite-specific):")
print(implicit_coercion)

### Integer Division Trap

In [ ]:
# The integer division trap
int_div = q("""
SELECT
    7 / 2                   AS integer_division,   -- gives 3, not 3.5
    7.0 / 2                 AS float_division,     -- gives 3.5
    CAST(7 AS REAL) / 2     AS cast_float_div,     -- also 3.5
    ROUND(7.0 / 2, 2)       AS rounded             -- 3.5
""")
print("Division behaviour:")
print(int_div)

### Date Functions in SQLite

In [ ]:
date_ops = q("""
SELECT
    signup_date,
    STRFTIME('%Y', signup_date)     AS year,
    STRFTIME('%m', signup_date)     AS month,
    STRFTIME('%Y-%m', signup_date)  AS year_month,
    CAST(julianday('2024-01-01') - julianday(signup_date) AS INTEGER) AS days_since_signup
FROM users
WHERE user_id <= 5
""")
print("Date operations:")
print(date_ops)

### Signups by Month

In [ ]:
# Time-series: monthly signups
monthly_signups = q("""
SELECT
    STRFTIME('%Y-%m', signup_date) AS month,
    COUNT(*) AS new_users
FROM   users
GROUP BY STRFTIME('%Y-%m', signup_date)
ORDER BY month
""")
print("Monthly user signups:")
print(monthly_signups)

### Comparing Date Functions Across Databases

### CASE WHEN — Conditional Expressions

In [ ]:
# CASE WHEN is SQL's if/elif/else
case_demo = q("""
SELECT
    watch_minutes,
    CASE
        WHEN watch_minutes < 30        THEN 'short'
        WHEN watch_minutes BETWEEN 30 AND 89 THEN 'medium'
        WHEN watch_minutes >= 90       THEN 'long'
        ELSE 'unknown'
    END AS session_length
FROM watch_events
LIMIT 5
""")
print(case_demo)

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

In [ ]:
signup_trend = q("""
SELECT
    STRFTIME('%Y-%m', signup_date)                          AS month,
    SUM(CASE WHEN plan = 'Premium' THEN 1 ELSE 0 END)      AS premium,
    SUM(CASE WHEN plan = 'Basic'   THEN 1 ELSE 0 END)      AS basic,
    SUM(CASE WHEN plan = 'Free'    THEN 1 ELSE 0 END)      AS free,
    COUNT(*)                                                AS total
FROM   users
WHERE  signup_date >= '2022-01-01'
  AND  signup_date <  '2024-01-01'
GROUP BY STRFTIME('%Y-%m', signup_date)
ORDER BY month
""")
print("Monthly signups by plan (2022–2023):")
print(signup_trend.to_string(index=False))

In [ ]:
tenure_segments = q("""
WITH tenure AS (
    SELECT
        user_id,
        plan,
        signup_date,
        CAST(julianday('2024-01-01') - julianday(signup_date) AS INTEGER) AS days_tenure
    FROM users
    WHERE churned = 0
)
SELECT
    CASE
        WHEN days_tenure >= 730 THEN '2yr+ veteran'
        WHEN days_tenure >= 365 THEN '1-2yr loyal'
        WHEN days_tenure >= 180 THEN '6-12mo growing'
        ELSE 'under 6mo new'
    END AS tenure_band,
    COUNT(*)            AS user_count,
    SUM(CASE plan WHEN 'Premium' THEN 12.90 WHEN 'Basic' THEN 8.90 ELSE 0 END) AS monthly_rev
FROM tenure
GROUP BY tenure_band
ORDER BY MIN(days_tenure) DESC
""")
print("\nUser tenure segmentation:")
print(tenure_segments.to_string(index=False))

In [ ]:
day_of_week = q("""
SELECT
    CAST(STRFTIME('%w', watch_started) AS INTEGER) AS day_num,
    CASE STRFTIME('%w', watch_started)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS day_name,
    COUNT(*)                            AS sessions,
    ROUND(AVG(watch_minutes), 1)        AS avg_minutes
FROM   watch_events
GROUP BY STRFTIME('%w', watch_started)
ORDER BY day_num
""")
print("\nWatch activity by day of week:")
print(day_of_week.to_string(index=False))

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
result = q("""
SELECT
    STRFTIME('%Y', signup_date) AS year,
    CASE 
        WHEN STRFTIME('%m', signup_date) IN ('01','02','03') THEN 'Q1'
        WHEN STRFTIME('%m', signup_date) IN ('04','05','06') THEN 'Q2'
        WHEN STRFTIME('%m', signup_date) IN ('07','08','09') THEN 'Q3'
        ELSE 'Q4'
    END AS quarter,
    COUNT(*) AS signups
FROM users
GROUP BY year, quarter
ORDER BY year, quarter
""")
print(result)

```sql
SELECT 
    100 * SUM(CASE WHEN plan = 'Premium' THEN 1 ELSE 0 END) / COUNT(*) AS premium_pct
FROM users
```

In [ ]:
bug_demo = q("""
SELECT 
    SUM(CASE WHEN plan = 'Premium' THEN 1 ELSE 0 END) AS premium_count,
    COUNT(*) AS total,
    100 * SUM(CASE WHEN plan = 'Premium' THEN 1 ELSE 0 END) / COUNT(*) AS bad_pct,
    ROUND(100.0 * SUM(CASE WHEN plan = 'Premium' THEN 1 ELSE 0 END) / COUNT(*), 1) AS good_pct
FROM users
""")
print(bug_demo)

In [ ]:
result = q("""
SELECT
    STRFTIME('%Y-%m', watch_started)            AS month,
    COUNT(*)                                    AS sessions,
    ROUND(AVG(watch_minutes), 1)                AS avg_min,
    CASE WHEN AVG(watch_minutes) > 76 THEN 'high' ELSE 'normal' END AS engagement_flag
FROM   watch_events
WHERE  watch_minutes IS NOT NULL
GROUP BY STRFTIME('%Y-%m', watch_started)
ORDER BY month
""")
print(result.to_string(index=False))

---

# Chapter 40: Indexes and Query Performance

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd
import time

# Use a larger dataset to illustrate performance
conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

### Creating Indexes

In [ ]:
# Check existing indexes
existing_indexes = q("""
SELECT name, tbl_name, sql
FROM   sqlite_master
WHERE  type = 'index'
ORDER BY tbl_name
""")
print("Existing indexes:")
print(existing_indexes)

In [ ]:
# Create indexes on frequently-queried columns
cur = conn.cursor()

# Single-column index: for WHERE plan = 'Premium' queries
cur.execute("CREATE INDEX idx_users_plan ON users(plan)")

# Single-column index: for JOIN ON user_id in watch_events
cur.execute("CREATE INDEX idx_events_user_id ON watch_events(user_id)")

# Composite index: for WHERE country = 'SG' AND churned = 0
cur.execute("CREATE INDEX idx_users_country_churned ON users(country, churned)")

conn.commit()
print("Indexes created.")

# Verify
indexes = q("""
SELECT name, tbl_name
FROM   sqlite_master
WHERE  type = 'index'
ORDER BY tbl_name, name
""")
print(indexes)

### Reading EXPLAIN

In [ ]:
# EXPLAIN QUERY PLAN in SQLite (equivalent to EXPLAIN in PostgreSQL)
plan = q("""
EXPLAIN QUERY PLAN
SELECT user_id, name, plan
FROM   users
WHERE  plan = 'Premium'
""")
print("Query plan WITH index on plan:")
print(plan)

In [ ]:
# Drop the index to show the difference
cur.execute("DROP INDEX idx_users_plan")
conn.commit()

plan_no_index = q("""
EXPLAIN QUERY PLAN
SELECT user_id, name, plan   -- same query, but idx_users_plan is gone
FROM   users
WHERE  plan = 'Premium'
""")
print("\nQuery plan WITHOUT index on plan:")
print(plan_no_index)

In [ ]:
# Re-create the index
cur.execute("CREATE INDEX idx_users_plan ON users(plan)")
conn.commit()

### Composite Index Column Order Matters

In [ ]:
# Composite index: country first, churned second
# It can be used for:
# WHERE country = 'SG'  (leading column)
# WHERE country = 'SG' AND churned = 0  (both columns)
# But NOT for: WHERE churned = 0 (non-leading column alone)

plan_composite = q("""
EXPLAIN QUERY PLAN
SELECT user_id, name 
FROM   users
WHERE  country = 'SG' AND churned = 0
""")
print("Composite index used (country + churned):")
print(plan_composite)

### Timing Queries (Simulated)

In [ ]:
import time

def timed_query(sql: str, label: str) -> None:
    start = time.perf_counter()
    result = pd.read_sql(sql, conn)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"{label}: {len(result)} rows in {elapsed:.2f}ms")

# Simple filter — benefits from index
timed_query(
    "SELECT * FROM users WHERE plan = 'Premium'",
    "Filter by plan (indexed)"
)

# Join — benefits from index on FK
timed_query("""
SELECT u.name, we.watch_minutes
FROM   watch_events we
JOIN   users u ON we.user_id = u.user_id
WHERE  u.plan = 'Premium'
""", "Join with index on user_id + plan")

### Common Query Patterns and Index Strategies

In [ ]:
# Query patterns and index recommendations
patterns = [
    ("WHERE col = val", "Single-column index on col"),
    ("WHERE col_a = v AND col_b = v", "Composite index (col_a, col_b) — put most selective first"),
    ("JOIN a ON a.fk = b.pk", "Index on the FK column (PK is already indexed)"),
    ("ORDER BY col LIMIT n", "Index on col for index-ordered scan"),
    ("WHERE col LIKE '%pattern%'", "No index can help — full scan required"),
    ("WHERE func(col) = val", "No index on the column — index is on the raw value"),
]

df_patterns = pd.DataFrame(patterns, columns=["Query Pattern", "Index Recommendation"])
print("Index strategy guide:")
print(df_patterns.to_string(index=False))

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

cur = conn.cursor()

In [ ]:
problem_query = """
SELECT
    u.country,
    COUNT(DISTINCT we.movie_id) AS distinct_movies,
    ROUND(100.0 * SUM(we.completed) / COUNT(*), 1) AS completion_pct
FROM   watch_events we
JOIN   users u ON we.user_id = u.user_id
GROUP BY u.country
HAVING COUNT(DISTINCT we.movie_id) >= 5
ORDER BY completion_pct DESC
"""

plan = q("EXPLAIN QUERY PLAN " + problem_query)
print("Query plan BEFORE indexing:")
print(plan)

In [ ]:
# watch_events joins on user_id — index it
cur.execute("CREATE INDEX idx_we_user_id ON watch_events(user_id)")
# users joins and groups by country — index it
cur.execute("CREATE INDEX idx_users_country ON users(country)")
conn.commit()

plan_after = q("EXPLAIN QUERY PLAN " + problem_query)
print("\nQuery plan AFTER indexing:")
print(plan_after)

In [ ]:
# Carlos's pre-deployment checklist for any query
def production_check(sql: str, label: str) -> None:
    print(f"\n=== {label} ===")
    
    # 1. EXPLAIN first
    plan = q("EXPLAIN QUERY PLAN " + sql)
    has_scan = "SCAN" in " ".join(plan["detail"].tolist())
    print(f"Plan: {'⚠ FULL SCAN detected' if has_scan else 'Index-based lookup'}")
    
    # 2. Row count
    count_sql = f"SELECT COUNT(*) AS n FROM ({sql})"
    try:
        n = q(count_sql)['n'][0]
        print(f"Result rows: {n}")
    except Exception:
        print("Row count: N/A (complex query)")
    
    # 3. Time
    import time
    start = time.perf_counter()
    result = q(sql)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"Time: {elapsed:.1f}ms (on {len(result)}-row sample)")

production_check(
    "SELECT user_id, COUNT(*) AS sessions FROM watch_events GROUP BY user_id ORDER BY sessions DESC LIMIT 10",
    "Top viewers query"
)

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
audit_query = """
SELECT 
    u.plan, 
    COUNT(we.event_id) AS sessions
FROM   watch_events we
JOIN   users u ON we.user_id = u.user_id
WHERE  we.country = 'SG'
GROUP BY u.plan
"""

plan_before = q("EXPLAIN QUERY PLAN " + audit_query)
print("Before:")
print(plan_before)

In [ ]:
# Check before
plan_before = q("EXPLAIN QUERY PLAN " + audit_query)
print("Before indexing:")
print(plan_before)

# Add index on watch_events.country
cur.execute("CREATE INDEX idx_we_country ON watch_events(country)")
conn.commit()

# Check after
plan_after = q("EXPLAIN QUERY PLAN " + audit_query)
print("\nAfter indexing watch_events.country:")
print(plan_after)

# Run the query
result = q(audit_query)
print("\nResult:")
print(result)

---

# Chapter 41: DDL, DML, and Views — The Write Side of SQL

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)

def execute(sql: str) -> None:
    conn.execute(sql)
    conn.commit()

### CREATE TABLE with Constraints

In [ ]:
execute("""
CREATE TABLE IF NOT EXISTS subscription_events (
    event_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id         INTEGER NOT NULL,
    event_type      TEXT    NOT NULL CHECK (event_type IN ('subscribe','upgrade','downgrade','cancel')),
    from_plan       TEXT,
    to_plan         TEXT    NOT NULL,
    event_timestamp TIMESTAMP NOT NULL DEFAULT (datetime('now')),
    amount_sgd      REAL    NOT NULL CHECK (amount_sgd >= 0)
)
""")

# Verify
schema = q("SELECT sql FROM sqlite_master WHERE name = 'subscription_events'")
print("Created table schema:")
print(schema['sql'][0])

### INSERT INTO

In [ ]:
# Insert single row. We pass event_timestamp explicitly so the demo output is
# reproducible; omit it and the column's DEFAULT (datetime('now')) fills it in.
execute("""
INSERT INTO subscription_events (user_id, event_type, from_plan, to_plan, amount_sgd, event_timestamp)
VALUES (1, 'subscribe', NULL, 'Free', 0.0, '2025-01-05 09:00:00')
""")

# Insert multiple rows
execute("""
INSERT INTO subscription_events (user_id, event_type, from_plan, to_plan, amount_sgd, event_timestamp)
VALUES
    (1, 'upgrade',   'Free',    'Premium', 12.90, '2025-01-06 10:15:00'),
    (2, 'subscribe', NULL,      'Basic',    8.90, '2025-01-07 11:30:00'),
    (3, 'subscribe', NULL,      'Free',     0.00, '2025-01-08 12:45:00'),
    (3, 'upgrade',   'Free',    'Basic',    8.90, '2025-01-09 14:00:00'),
    (2, 'cancel',    'Basic',   'Free',     0.00, '2025-01-10 15:20:00')
""")

print(q("SELECT * FROM subscription_events ORDER BY event_id"))

### UPDATE

In [ ]:
# Update a specific row
execute("""
UPDATE subscription_events
SET    amount_sgd = 12.90
WHERE  event_type = 'upgrade' AND to_plan = 'Premium'
""")

result = q("""
SELECT event_id, user_id, event_type, to_plan, amount_sgd
FROM   subscription_events
WHERE  event_type = 'upgrade'
""")
print("After UPDATE:")
print(result)

In [ ]:
# UPDATE with subquery — update all users whose plan changed in the events table
execute("""
CREATE TABLE IF NOT EXISTS demo_users (
    user_id  INTEGER PRIMARY KEY,
    name     TEXT,
    plan     TEXT
)
""")
execute("INSERT INTO demo_users VALUES (1, 'Ravi Kumar', 'Free')")
execute("INSERT INTO demo_users VALUES (2, 'Siti Rahman', 'Basic')")
execute("INSERT INTO demo_users VALUES (3, 'Nguyen', 'Free')")

# Sync plan from latest subscription event
execute("""
UPDATE demo_users
SET    plan = (
    SELECT to_plan
    FROM   subscription_events se
    WHERE  se.user_id = demo_users.user_id
    ORDER BY event_timestamp DESC
    LIMIT 1
)
WHERE  user_id IN (SELECT DISTINCT user_id FROM subscription_events)
""")

print("After syncing plans:")
print(q("SELECT * FROM demo_users"))

### DELETE

In [ ]:
# Delete rows matching a condition
execute("""
DELETE FROM subscription_events
WHERE event_type = 'subscribe' AND to_plan = 'Free' AND amount_sgd = 0.0
""")

count = q("SELECT COUNT(*) AS n FROM subscription_events")['n'][0]
print(f"After deleting free signups: {count} rows remain")
print(q("SELECT * FROM subscription_events ORDER BY event_id"))

### ALTER TABLE

In [ ]:
# Add a new column
execute("ALTER TABLE subscription_events ADD COLUMN currency TEXT DEFAULT 'SGD'")
print("After adding currency column:")
print(q("SELECT * FROM subscription_events ORDER BY event_id LIMIT 2"))

### CREATE VIEW

In [ ]:
# Load CinemaStream data for view examples
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

# Create a view that simplifies the most common join
execute("""
CREATE VIEW IF NOT EXISTS enriched_events AS
SELECT
    we.event_id,
    we.user_id,
    u.name          AS user_name,
    u.plan,
    u.country       AS user_country,
    m.title         AS movie_title,
    m.genre,
    we.watch_minutes,
    we.completed,
    we.device,
    we.watch_started
FROM   watch_events we
JOIN   users  u ON we.user_id  = u.user_id
JOIN   movies m ON we.movie_id = m.movie_id
""")

# Now queries are much simpler
simple_query = q("""
SELECT user_name, plan, movie_title, watch_minutes, device
FROM   enriched_events
WHERE  plan = 'Premium' AND completed = 1
ORDER BY watch_minutes DESC
LIMIT 5
""")
print("Query via view:")
print(simple_query)

## 3. CinemaStream in Practice

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

def q(sql: str) -> pd.DataFrame:
    return pd.read_sql(sql, conn)
def execute(sql: str) -> None:
    conn.execute(sql)
    conn.commit()

In [ ]:
execute("""
CREATE TABLE IF NOT EXISTS watch_events_staging (
    raw_event_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id        INTEGER,
    movie_id       INTEGER,
    watch_started  TEXT,
    watch_minutes  INTEGER,
    completed      INTEGER,
    device         TEXT,
    country        TEXT,
    ingested_at    TIMESTAMP DEFAULT (datetime('now')),
    is_valid       INTEGER DEFAULT 1,
    error_reason   TEXT
)
""")
print("Staging table created.")

In [ ]:
# Simulate raw batch with one bad record
import datetime

raw_events = [
    (201, 101, '2024-01-10 20:00:00', 95, 1, 'TV', 'SG'),
    (202, 102, '2024-01-10 20:30:00', 112, 1, 'Mobile', 'MY'),
    (9999, 101, '2024-01-10 21:00:00', 45, 0, 'Web', 'VN'),  # invalid user_id
    (201, 999, '2024-01-10 22:00:00', -5, 1, 'TV', 'SG'),     # invalid movie_id + negative minutes
]

for user_id, movie_id, watch_started, watch_minutes, completed, device, country in raw_events:
    # Validate
    is_valid = 1
    error_reason = None

    # Check: user must exist
    user_exists = q(f"SELECT COUNT(*) AS n FROM users WHERE user_id = {user_id}")['n'][0]
    if not user_exists:
        is_valid, error_reason = 0, f"user_id {user_id} not in users table"

    # Check: watch_minutes must be positive
    elif watch_minutes <= 0:
        is_valid, error_reason = 0, f"watch_minutes={watch_minutes} must be positive"

    conn.execute("""
    INSERT INTO watch_events_staging
    (user_id, movie_id, watch_started, watch_minutes, completed, device, country, is_valid, error_reason)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (user_id, movie_id, watch_started, watch_minutes, completed, device, country, is_valid, error_reason))

conn.commit()

staging = q("SELECT * FROM watch_events_staging")
print("Staging table contents:")
print(staging[['raw_event_id', 'user_id', 'movie_id', 'watch_minutes', 'is_valid', 'error_reason']])

In [ ]:
execute("""
INSERT INTO watch_events (user_id, movie_id, watch_started, watch_minutes, completed, device, country)
SELECT user_id, movie_id, watch_started, watch_minutes, completed, device, country
FROM   watch_events_staging
WHERE  is_valid = 1
""")

promoted_count = q("""
SELECT COUNT(*) AS n FROM watch_events
WHERE watch_started >= '2024-01-10'
""")['n'][0]
print(f"Records promoted to production: {promoted_count}")

# Create analyst view over clean events
execute("""
CREATE VIEW IF NOT EXISTS analyst_events AS
SELECT
    we.event_id,
    u.name       AS user_name,
    u.plan,
    u.country    AS user_country,
    m.title      AS movie,
    m.genre,
    we.watch_minutes,
    we.completed,
    we.device
FROM   watch_events we
JOIN   users  u ON we.user_id  = u.user_id
JOIN   movies m ON we.movie_id = m.movie_id
""")
print("analyst_events view created — analysts can query this without knowing the schema.")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
execute("""
CREATE TABLE IF NOT EXISTS ratings (
    rating_id  INTEGER  PRIMARY KEY AUTOINCREMENT,
    user_id    INTEGER  NOT NULL REFERENCES users(user_id),
    movie_id   INTEGER  NOT NULL REFERENCES movies(movie_id),
    stars      INTEGER  NOT NULL CHECK (stars BETWEEN 1 AND 5),
    rated_at   TIMESTAMP NOT NULL DEFAULT (datetime('now')),
    UNIQUE (user_id, movie_id)
)
""")
print("ratings table created.")

# Verify the constraint works
try:
    conn.execute("INSERT INTO ratings (user_id, movie_id, stars) VALUES (1, 101, 5)")
    conn.execute("INSERT INTO ratings (user_id, movie_id, stars) VALUES (1, 101, 3)")  # duplicate
    conn.commit()
except Exception as e:
    print(f"Duplicate rejected: {e}")

In [ ]:
execute("""
CREATE TABLE IF NOT EXISTS demo_users2 (
    user_id    INTEGER PRIMARY KEY,
    name       TEXT,
    plan       TEXT,
    deleted_at TIMESTAMP
)
""")
execute("INSERT INTO demo_users2 VALUES (1, 'Ravi Kumar', 'Premium', NULL)")
execute("INSERT INTO demo_users2 VALUES (2, 'Siti Rahman', 'Basic', NULL)")
execute("INSERT INTO demo_users2 VALUES (3, 'Nguyen Van Minh', 'Free', NULL)")

# Soft delete user 3
execute("UPDATE demo_users2 SET deleted_at = datetime('now') WHERE user_id = 3")

# Query active users only
active = q("SELECT user_id, name, plan FROM demo_users2 WHERE deleted_at IS NULL")
print("Active users (soft-deleted excluded):")
print(active)

In [ ]:
execute("""
CREATE VIEW IF NOT EXISTS country_metrics AS
SELECT
    country,
    COUNT(*)                                                              AS total_users,
    SUM(CASE WHEN churned = 0 THEN 1 ELSE 0 END)                        AS active_users,
    SUM(CASE WHEN plan = 'Premium' AND churned = 0 THEN 1 ELSE 0 END)   AS premium_users,
    SUM(CASE WHEN plan = 'Basic'   AND churned = 0 THEN 1 ELSE 0 END)   AS basic_users,
    SUM(CASE WHEN plan = 'Free'    AND churned = 0 THEN 1 ELSE 0 END)   AS free_users,
    ROUND(
        SUM(CASE WHEN plan = 'Premium' AND churned = 0 THEN 12.90 ELSE 0 END) +
        SUM(CASE WHEN plan = 'Basic'   AND churned = 0 THEN  8.90 ELSE 0 END), 2
    )                                                                    AS monthly_rev_sgd
FROM users
GROUP BY country
""")

top_countries = q("""
SELECT country, active_users, premium_users, basic_users, monthly_rev_sgd
FROM   country_metrics
ORDER BY monthly_rev_sgd DESC
LIMIT 3
""")
print("Top 3 countries by monthly revenue:")
print(top_countries)

---

# Chapter 41a: NoSQL Databases — When the Table Doesn't Fit

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

### MongoDB — document CRUD

In [ ]:
# pip install mongomock  (drop-in pymongo replacement for testing)
import mongomock

client = mongomock.MongoClient()
db = client["catalog"]
products = db["products"]

# Insert — schema is flexible per document
products.insert_many([
    {"name": "The Pragmatic Programmer", "category": "book",
     "isbn": "978-0-13-595705-9", "price_usd": 49.99},
    {"name": "Winter Jacket", "category": "clothing",
     "size": ["S", "M", "L", "XL"], "price_usd": 89.99},
    {"name": "Smart Bulb", "category": "electronics",
     "wattage": 9, "wifi": True, "price_usd": 14.99},
])

print(f"Total documents: {products.count_documents({})}")

# Query — filter by field
books = list(products.find({"category": "book"}, {"_id": 0, "name": 1, "isbn": 1}))
print("Books:", books)

# Query — range filter
affordable = list(products.find(
    {"price_usd": {"$lt": 20}},
    {"_id": 0, "name": 1, "price_usd": 1}
))
print("Under $20:", affordable)

In [ ]:
import mongomock

client = mongomock.MongoClient()
products = client["catalog"]["products"]
products.insert_many([
    {"name": "Smart Bulb", "category": "electronics", "price_usd": 14.99},
    {"name": "LED Strip", "category": "electronics", "price_usd": 9.99},
])

# Update — increment price by 10%
# (Real MongoDB: {"$mul": {"price_usd": 1.10}} — mongomock testing library doesn't support $mul,
#  so we achieve the same result with a Python loop + $set for local QC runs)
for doc in list(products.find({"category": "electronics"})):
    products.update_one({"_id": doc["_id"]}, {"$set": {"price_usd": round(doc["price_usd"] * 1.10, 2)}})

for p in products.find({"category": "electronics"}, {"_id": 0}):
    print(p)

# Delete
products.delete_one({"name": "LED Strip"})
print(f"After delete: {products.count_documents({})} document(s)")

### MongoDB — safe query wrappers

In [ ]:
import mongomock
from typing import Optional

client = mongomock.MongoClient()
users = client["app"]["users"]
users.insert_many([
    {"username": "priya", "role": "admin"},
    {"username": "carlos", "role": "engineer"},
])


def find_user_safe(collection, username: str) -> Optional[dict]:
    """Use a parameterized filter — never f-string user input into a query."""
    return collection.find_one({"username": username}, {"_id": 0})


def find_user_unsafe(collection, username: str) -> str:
    """NEVER do this — hypothetical injection risk."""
    return f"db.users.find({{username: '{username}'}})"


result = find_user_safe(users, "priya")
print("Safe lookup:", result)

injection_attempt = "'; db.users.drop(); var x = '"
print("Safe with injection attempt:", find_user_safe(users, injection_attempt))
print("Still have users:", users.count_documents({}))

### Redis — key-value operations

In [ ]:
# pip install fakeredis  (in-memory Redis for testing)
import fakeredis

r = fakeredis.FakeRedis(decode_responses=True)

# String set/get
r.set("greeting", "hello")
print(r.get("greeting"))

# TTL — key expires after N seconds
r.set("temp_token", "abc123", ex=60)
print(f"Token exists: {r.exists('temp_token')}")
print(f"TTL remaining: {r.ttl('temp_token')}s")

# Counter (atomic — safe for concurrent requests)
r.set("page_views", 0)
r.incr("page_views")
r.incr("page_views")
r.incr("page_views")
print(f"Page views: {r.get('page_views')}")

In [ ]:
import fakeredis

r = fakeredis.FakeRedis(decode_responses=True)

# Hash — store a structured object as a single key
r.hset("user:101", mapping={
    "name":        "Priya",
    "last_movie":  "507",
    "resume_sec":  "1423",
    "device":      "mobile",
})

# Retrieve the whole hash
user_data = r.hgetall("user:101")
print(user_data)

# Retrieve one field
print(f"Priya's resume position: {r.hget('user:101', 'resume_sec')}s")

# Update one field
r.hset("user:101", "resume_sec", "1680")
print(f"Updated position: {r.hget('user:101', 'resume_sec')}s")

### SQL vs NoSQL — decision table in code

In [ ]:
from dataclasses import dataclass
from typing import Literal

@dataclass
class DataStoreRequirements:
    schema_varies_per_record: bool
    need_sub_ms_reads: bool
    data_can_be_lost_on_restart: bool
    need_acid_transactions: bool
    heavy_cross_collection_joins: bool
    complex_aggregation_queries: bool


def choose_store(req: DataStoreRequirements) -> Literal["SQL", "MongoDB", "Redis"]:
    if req.need_sub_ms_reads and req.data_can_be_lost_on_restart:
        return "Redis"
    if req.need_acid_transactions or req.heavy_cross_collection_joins:
        return "SQL"
    if req.complex_aggregation_queries:
        return "SQL"
    if req.schema_varies_per_record:
        return "MongoDB"
    return "SQL"   # default — SQL unless there is a specific reason not to


# Test cases
cases = [
    ("Resume position cache", DataStoreRequirements(
        schema_varies_per_record=False, need_sub_ms_reads=True,
        data_can_be_lost_on_restart=True, need_acid_transactions=False,
        heavy_cross_collection_joins=False, complex_aggregation_queries=False)),
    ("Product catalog (variable fields)", DataStoreRequirements(
        schema_varies_per_record=True, need_sub_ms_reads=False,
        data_can_be_lost_on_restart=False, need_acid_transactions=False,
        heavy_cross_collection_joins=False, complex_aggregation_queries=False)),
    ("Subscription billing", DataStoreRequirements(
        schema_varies_per_record=False, need_sub_ms_reads=False,
        data_can_be_lost_on_restart=False, need_acid_transactions=True,
        heavy_cross_collection_joins=True, complex_aggregation_queries=True)),
]

for name, req in cases:
    print(f"{name:40s} → {choose_store(req)}")

## 3. CinemaStream in Practice

In [ ]:
# cinemastream/scripts/session_cache.py
"""
Redis-based session cache for CinemaStream's "continue watching" feature.
Stores and retrieves the user's resume position per movie.

Run: python cinemastream/scripts/session_cache.py
Requires: pip install fakeredis (or a real Redis instance with redis-py)
"""

from dataclasses import dataclass
from typing import Optional
import fakeredis        # swap for: import redis; r = redis.Redis(host="localhost")


# In production: r = redis.Redis(host="redis.cinemastream.internal", port=6379, decode_responses=True)
r = fakeredis.FakeRedis(decode_responses=True)

SESSION_TTL_SECONDS = 60 * 60 * 24 * 7   # 7 days — resume persists across devices


@dataclass
class ResumeState:
    user_id: int
    movie_id: int
    resume_sec: int
    duration_sec: int
    device: str


def _session_key(user_id: int, movie_id: int) -> str:
    return f"resume:{user_id}:{movie_id}"


def save_resume(state: ResumeState) -> None:
    """Write resume position. Called on pause/stop events."""
    key = _session_key(state.user_id, state.movie_id)
    r.hset(key, mapping={
        "resume_sec":  str(state.resume_sec),
        "duration_sec": str(state.duration_sec),
        "device":      state.device,
    })
    r.expire(key, SESSION_TTL_SECONDS)


def get_resume(user_id: int, movie_id: int) -> Optional[ResumeState]:
    """Read resume position. Called on play events."""
    key = _session_key(user_id, movie_id)
    data = r.hgetall(key)
    if not data:
        return None
    return ResumeState(
        user_id=user_id,
        movie_id=movie_id,
        resume_sec=int(data["resume_sec"]),
        duration_sec=int(data["duration_sec"]),
        device=data["device"],
    )


def clear_resume(user_id: int, movie_id: int) -> None:
    """Delete resume when user finishes or manually restarts."""
    r.delete(_session_key(user_id, movie_id))


def main() -> None:
    # Priya pauses "Hujan di Singapura" at 23 minutes on her phone
    save_resume(ResumeState(
        user_id=1, movie_id=101, resume_sec=1380,
        duration_sec=5640, device="mobile"
    ))
    print("Saved resume for user 1, movie 101.")

    # She opens the app on her smart TV — picks up where she left off
    state = get_resume(user_id=1, movie_id=101)
    if state:
        pct = state.resume_sec / state.duration_sec * 100
        print(f"Resuming movie {state.movie_id} at {state.resume_sec}s "
              f"({pct:.0f}%) on {state.device}.")
    else:
        print("No resume position found — starting from beginning.")

    # She finishes the movie — clear the resume
    clear_resume(user_id=1, movie_id=101)
    print(f"Movie finished. Resume cleared. "
          f"Key exists: {r.exists(_session_key(1, 101))}")

    # User 2 has no saved position
    state2 = get_resume(user_id=2, movie_id=101)
    print(f"User 2 resume: {state2}")


if __name__ == "__main__":
    main()

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import fakeredis

r = fakeredis.FakeRedis(decode_responses=True)
RATE_LIMIT = 100
WINDOW_SECONDS = 60


def check_rate_limit(api_key: str) -> bool:
    """Return True if the request is allowed; False if rate limit exceeded."""
    counter_key = f"rate:{api_key}"
    current = r.incr(counter_key)
    if current == 1:
        r.expire(counter_key, WINDOW_SECONDS)
    return current <= RATE_LIMIT


# Simulate 101 requests
allowed = sum(1 for _ in range(101) if check_rate_limit("key_abc"))
blocked = 101 - allowed
print(f"Allowed: {allowed}, Blocked: {blocked}")
print(f"101st request allowed: {check_rate_limit('key_abc')}")

In [ ]:
import fakeredis

r = fakeredis.FakeRedis(decode_responses=True)
MAX_RECENT = 5


def record_watch(user_id: int, movie_id: int) -> None:
    key = f"recent:{user_id}"
    r.lpush(key, str(movie_id))
    r.ltrim(key, 0, MAX_RECENT - 1)   # keep only the 5 most recent


def get_recently_watched(user_id: int) -> list[int]:
    key = f"recent:{user_id}"
    return [int(m) for m in r.lrange(key, 0, -1)]


# User 1 watches 8 movies in sequence
for movie_id in [10, 20, 30, 40, 50, 60, 70, 80]:
    record_watch(user_id=1, movie_id=movie_id)

recent = get_recently_watched(user_id=1)
print(f"Recently watched (most recent first): {recent}")
print(f"Count: {len(recent)} (expect 5)")